# Agentic RAG Schedule Assistant — Colab dev/test notebook

This notebook lets you build, test, and (optionally) temporarily expose the FastAPI
agent from Google Colab using `pyngrok`, before deploying it permanently on Render.

Steps: clone the repo -> install deps -> (optional) set `GEMINI_API_KEY` -> run the
server -> test with example queries -> optionally tunnel with ngrok for a shareable URL.

In [ ]:
# 1. Clone your repo (replace with your GitHub URL after you push this project)
!git clone https://github.com/<your-username>/schedule-agent.git
%cd schedule-agent

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt pyngrok

In [ ]:
# 3. (Optional) Set your Gemini API key for the full agentic function-calling agent.
# Without this, the app automatically falls back to a rule-based router so it still works.
import os
os.environ["GEMINI_API_KEY"] = ""  # paste your key here, or leave blank for fallback mode

In [ ]:
# 4. Generate the 30-day sample schedule and build the ChromaDB vector index
!python data/generate_schedule.py
from app.vectorstore import get_store
store = get_store()
print("Indexed events:", len(store.get_all()["ids"]))

In [ ]:
# 5. Try the agent directly in-notebook (no server needed)
from app.agent import get_agent
agent = get_agent()

for q in [
    "What do I have scheduled tomorrow?",
    "Am I free Friday afternoon?",
    "Add a meeting on August 15 at 3 PM.",
    "Move my meeting from 3 PM to 4 PM on August 15.",
]:
    r = agent.chat(q)
    print(f"> {q}\n{r['reply']}\n")

In [ ]:
# 6. Run the FastAPI server in the background
import subprocess, time
server = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)
print("Server started")

In [ ]:
# 7. (Optional) Expose it publicly with ngrok for a quick shareable link
# Get a free authtoken at https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok
ngrok.set_auth_token("<your-ngrok-authtoken>")
public_url = ngrok.connect(8000)
print("Playground:", f"{public_url}/agent/playground")

## Deploying permanently to Render
Once you're happy with the agent, push this repo to GitHub and deploy it on
[Render](https://render.com) as a Web Service — see `README.md` in the repo root
for the exact steps. Render gives you a stable URL like
`https://schedule-agent.onrender.com/agent/playground`, which is what you should
submit as the final deployed URL.